# Generate full volumes with the latent priors

A prior samples a normalised latent (4 × 128³ for CT, 4 × 64³ for MRI). The latent is de-normalised with `latent_stats.json` and decoded **once** by the CoRe-AE it was trained on, with no patches and no blending.

| Prior | Sampler |
|---|---|
| `core-ldm-jit-ct512-{liif,clif}`, `core-ldm-jit-mri256-{liif,clif}` | JiT, Heun, 50 steps (generative model of the paper) |
| `core-ldm-ddpm-ct512-{liif,clif}` | DDPM-ε, 1000 ancestral steps (example weights) |

Command line: `python inference/generate.py --help`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))          # not needed after `pip install -e .`
import torch
import matplotlib.pyplot as plt
from inference.generate import generate
from inference.common import three_planes

## 1. CT 512³, JiT priors

In [ ]:
ct = {p: generate(p, num=1, seed=91415, out_dir='outputs/generate')[0]
      for p in ('core-ldm-jit-ct512-liif', 'core-ldm-jit-ct512-clif')}
for p, r in ct.items():
    print(f"{p}: {r['seconds']:.0f} s, peak GPU {r['peak_gib'] or 0:.2f} GiB")
three_planes({'JiT + LIIF': ct['core-ldm-jit-ct512-liif']['volume'],
              'JiT + CLIF': ct['core-ldm-jit-ct512-clif']['volume']}, title='Generated CT 512³ (seed 91415)');

## 2. MRI 256³, JiT priors

In [ ]:
mri = {p: generate(p, num=1, seed=91415, out_dir='outputs/generate')[0]
       for p in ('core-ldm-jit-mri256-liif', 'core-ldm-jit-mri256-clif')}
for p, r in mri.items():
    print(f"{p}: {r['seconds']:.0f} s, peak GPU {r['peak_gib'] or 0:.2f} GiB")
three_planes({'JiT + LIIF': mri['core-ldm-jit-mri256-liif']['volume'],
              'JiT + CLIF': mri['core-ldm-jit-mri256-clif']['volume']}, title='Generated MRI 256³ (seed 91415)', modality='mri');

## 3. CT 512³, DDPM-ε example prior (1000 steps, slower)

In [ ]:
ddpm = generate('core-ldm-ddpm-ct512-liif', num=1, seed=91415, out_dir='outputs/generate')[0]
print(f"{ddpm['seconds']:.0f} s, peak GPU {ddpm['peak_gib'] or 0:.2f} GiB")
three_planes({'DDPM-ε + LIIF': ddpm['volume']}, title='Generated CT 512³ (seed 91415)');

Each sample is saved in `outputs/generate/` as NIfTI (CT in HU, 1 × 1 × 1.5 mm), its latent (`.npy`) and a PNG preview.
Other options: `generate(prior, num=4, seed=0, steps=100, method='euler', precision='fp32')`.